
# Pneumonia Classification XAI (Grad-CAM) - Dataset: chest_xray1
This notebook applies Grad-CAM to explain the predictions of the model trained on the `chest_xray1` dataset.


In [ ]:

import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import cv2


In [ ]:

img_height = 224
img_width = 224

# Reconstruct Model Architecture
base_model = tf.keras.applications.DenseNet121(
    weights=None,
    include_top=False,
    input_shape=(img_height, img_width, 3)
)

inputs = keras.Input(shape=(img_height, img_width, 3))
x = tf.keras.applications.densenet.preprocess_input(inputs)
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.2)(x)
outputs = keras.layers.Dense(1, activation='sigmoid')(x)
model = keras.Model(inputs, outputs)

# Load Weights
weights_path = r"c:\Users\prakh\Downloads\archive (1)\chest_xray1\chest_xray1_model_weights.weights.h5"
if os.path.exists(weights_path):
    model.load_weights(weights_path)
    print("Weights loaded successfully.")
else:
    print(f"Warning: Weights file not found at {weights_path}. Please train the model first.")


In [ ]:

def generate_gradcam(img_array, model, last_conv_layer_name, pred_index=None):
    # 1. Create a model that maps the input image to the activations of the last conv layer as well as the output predictions
    grad_model = keras.models.Model(
        model.inputs, [model.get_layer(last_conv_layer_name).output, model.output]
    )

    # 2. Compute the gradient of the top predicted class (or a specific class)
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # 3. This is the gradient of the output neuron (top predicted or chosen)
    # with regard to the output feature map of the last conv layer
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # 4. This is a vector where each entry is the mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # 5. We multiply each channel in the feature map array by "how important this channel is" with regard to the top predicted class
    # then sum all the channels to obtain the heatmap class activation
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # 6. For visualization purpose, we will also normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

def display_gradcam(img_path, heatmap, alpha=0.4):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (224, 224))

    # Rescale heatmap to a range 0-255
    heatmap = np.uint8(255 * heatmap)

    # Use jet colormap to colorize heatmap
    jet = plt.colormaps.get_cmap("jet")

    # Use RGB values of the colormap
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap]

    # Create an image with RGB colorized heatmap
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize((img.shape[1], img.shape[0]))
    jet_heatmap = keras.utils.img_to_array(jet_heatmap)

    # Superimpose the heatmap on original image
    superimposed_img = jet_heatmap * alpha + img
    superimposed_img = keras.utils.array_to_img(superimposed_img)

    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title("Original Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(superimposed_img)
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")
    plt.show()


In [ ]:

# Find the last conv layer name
# For DenseNet121, it's often 'relu' (the final activation of the last dense block)
last_conv_layer_name = 'relu' # Change if needed after checking model.summary()

# Test on an image from the test set
test_dir = r"c:\Users\prakh\Downloads\archive (1)\chest_xray1\pneumonia_images\test\PNEUMONIA"
if os.path.exists(test_dir):
    test_images = [os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith('.png')]
    if test_images:
        img_path = test_images[0]
        img = keras.utils.load_img(img_path, target_size=(img_height, img_width))
        img_array = keras.utils.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)

        heatmap = generate_gradcam(img_array, model, last_conv_layer_name)
        display_gradcam(img_path, heatmap)
    else:
        print("No images found in PNEUMONIA test directory.")
else:
    print(f"Directory not found: {test_dir}")
